In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 22 — Query Rewriting RAG Lab
## Rewrite Vague Questions Before Retrieval for Better Results

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from langchain_community.vectorstores import Chroma

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Knowledge base
docs = [
    Document(page_content="LangChain is a framework for building LLM applications. "
        "It provides chains, agents, and retrieval components.", metadata={"topic": "langchain"}),
    Document(page_content="ChromaDB is an open-source vector database for AI applications. "
        "It supports in-memory and persistent storage modes.", metadata={"topic": "chroma"}),
    Document(page_content="Ollama runs large language models locally. It supports Llama, "
        "Mistral, Qwen, and many other model families.", metadata={"topic": "ollama"}),
    Document(page_content="RAG retrieval quality depends on chunk size, overlap, embedding "
        "model choice, and the retrieval algorithm used.", metadata={"topic": "rag_quality"}),
    Document(page_content="Query rewriting transforms user queries into forms better suited "
        "for retrieval. Techniques include HyDE, step-back, and multi-query.", metadata={"topic": "rewriting"}),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="query_rewrite_lab")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Knowledge base indexed!")

## Step 2 — Baseline (No Rewriting)

In [ ]:
vague_queries = [
    "how do I make it work better?",       # Vague
    "that vector thing",                     # Ambiguous
    "what runs models on my laptop?",        # Informal
    "the python library for LLM apps",       # Imprecise
]

print("=== BASELINE (No Rewriting) ===")
for q in vague_queries:
    results = retriever.invoke(q)
    top_topic = results[0].metadata["topic"] if results else "none"
    print(f"  Q: {q!r:<45} → Top result: {top_topic}")

## Step 3 — Query Rewriting Strategies

In [ ]:
# Strategy 1: Simple clarification rewrite
clarify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite this vague query into a clear, specific search query. "
     "Add context that makes the query unambiguous. Return ONLY the rewritten query."),
    ("human", "{query}")
])
clarify_chain = clarify_prompt | llm | StrOutputParser()

# Strategy 2: HyDE (Hypothetical Document Embeddings)
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a short paragraph that would be the ideal answer to this "
     "question. This will be used to find similar documents. Write as if you are "
     "a technical document, not a conversational answer."),
    ("human", "{query}")
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

# Strategy 3: Multi-query expansion
multi_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 3 different versions of this search query that might "
     "retrieve relevant documents. Return one query per line, no numbering."),
    ("human", "{query}")
])
multi_chain = multi_prompt | llm | StrOutputParser()

# Strategy 4: Step-back prompting
stepback_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a more general, higher-level question that would help "
     "answer the specific question. Return ONLY the step-back question."),
    ("human", "{query}")
])
stepback_chain = stepback_prompt | llm | StrOutputParser()

print("Query rewriting strategies defined!")

## Step 4 — Compare All Strategies

In [ ]:
print("=== STRATEGY COMPARISON ===\n")

for q in vague_queries:
    print(f"Original: {q!r}")

    # Clarification
    clarified = clarify_chain.invoke({"query": q})
    results_c = retriever.invoke(clarified)
    print(f"  Clarified: {clarified!r}")
    print(f"    → {results_c[0].metadata['topic'] if results_c else 'none'}")

    # HyDE
    hypothetical = hyde_chain.invoke({"query": q})
    results_h = vectorstore.similarity_search(hypothetical, k=2)
    print(f"  HyDE doc: {hypothetical[:60]}...")
    print(f"    → {results_h[0].metadata['topic'] if results_h else 'none'}")

    # Multi-query
    multi = multi_chain.invoke({"query": q})
    sub_queries = [sq.strip() for sq in multi.strip().split("\n") if sq.strip()]
    all_results = set()
    for sq in sub_queries[:3]:
        for r in retriever.invoke(sq):
            all_results.add(r.metadata["topic"])
    print(f"  Multi-query ({len(sub_queries)} variants) → topics: {all_results}")

    # Step-back
    stepped = stepback_chain.invoke({"query": q})
    results_s = retriever.invoke(stepped)
    print(f"  Step-back: {stepped!r}")
    print(f"    → {results_s[0].metadata['topic'] if results_s else 'none'}")
    print()

## Step 5 — End-to-End Rewriting RAG Pipeline

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

def rewriting_rag(query, strategy="clarify"):
    """Full pipeline: rewrite → retrieve → answer."""
    if strategy == "clarify":
        rewritten = clarify_chain.invoke({"query": query})
    elif strategy == "hyde":
        rewritten = hyde_chain.invoke({"query": query})
    elif strategy == "stepback":
        rewritten = stepback_chain.invoke({"query": query})
    else:
        rewritten = query

    retrieved = retriever.invoke(rewritten)
    context = "\n".join([d.page_content for d in retrieved])

    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question using the provided context."),
        ("human", "Context: {context}\n\nQuestion: {question}")
    ])
    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({"context": context, "question": query})

    return {"rewritten": rewritten, "answer": answer, "sources": [d.metadata["topic"] for d in retrieved]}

# Test the full pipeline
result = rewriting_rag("how do I make retrieval better?", strategy="clarify")
print(f"Rewritten: {result['rewritten']}")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

## What You Learned
- **Query clarification** — rewrite vague queries
- **HyDE** — use hypothetical documents for better embedding match
- **Multi-query expansion** — cast a wider retrieval net
- **Step-back prompting** — generalize for broader context